<a href="https://colab.research.google.com/github/mayori-engineering/tec-mx-vault/blob/main/lab0_verificacion_entorno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 0 | Verificación de entorno
**Módulo 1 · sin puntos · 35 minutos · Modo precargado: sin API y sin costo**

## Contexto de negocio

Distribuidora Cuauhtémoc es la empresa que te acompañará todo el curso: consumo masivo, decenas de miles de tiendas del canal tradicional, y preventistas que reportan lo que ven en el anaquel. Sobre esos datos ejecutarás un flujo programado que ilustra este encargo del analista comercial: **"detecta el quiebre de anaquel más costoso de la última semana y prepara el caso para el gerente de zona"**.

Observa la forma del encargo: expresa un **objetivo**, pero este notebook no lo interpreta. Sus pasos están escritos de antemano. Esta diferencia conecta con el Aprende: ejecutar una ruta fija no demuestra que el sistema decida cuál ruta seguir.

**Tu papel en este laboratorio:** todavía no construyes nada. Verificas que tu entorno funciona antes de que existan entregas con puntos, y observas una traza didáctica de ruta fija, como preparación para auditar trazas en el Skill Lab 2. Si una celda falla, copia el mensaje de error completo y publícalo en el hilo de dudas de tu sección: detectar problemas de entorno esta semana es exactamente el propósito de este laboratorio.

## Objetivo

Al terminar este laboratorio habrás:

1. **Verificado tu entorno de trabajo**: los datos del curso cargan y pandas opera — una comprobación inicial, no una validación de todas las dependencias del curso.
2. **Observado una traza completa** de un flujo didáctico: del encargo comercial a una propuesta que debe revisar una persona.
3. **Aplicado las dos preguntas del curso** al código observado: quién determina los pasos y qué puede tocar el sistema.
4. **Propuesto un límite** que le pondrías antes de dejarlo operar sin supervisión — el criterio que refinarás cada semana.

## Cómo trabajar este notebook

- Ejecuta las celdas **en orden** (Entorno de ejecución, Ejecutar todas; o Shift+Enter celda por celda). No escribes código: solo la celda de respuestas es tuya.
- El flujo trabaja en **modo precargado**: las operaciones con datos se ejecutan, pero la secuencia está programada. No se ejecuta un modelo de IA ni se demuestra autonomía para planear.
- Entrega en Canvas un PDF con tus tres respuestas y evidencia de ejecución: el mensaje de carga de datos, la traza y el resultado de las comprobaciones técnicas. La revisión se hará sobre ese PDF, no sobre una liga editable. Se marca completo con la entrega; no se califica.
- **Alternativa sin Colab:** descarga este notebook y la carpeta de datos desde Canvas y ejecútalo en Jupyter local (guía en Requisitos técnicos).

### El mapa del laboratorio

```
  1. Preparar el       -->  2. Observar la traza     -->  3. Tus tres respuestas
     entorno                  del flujo                      (la unica celda tuya)
     (datos del curso)        (objetivo -> propuesta)
        \___________________ evidencia acumulada ___________________/
                                      |
                                      v
                     Verificador + pruebas  -->  entrega en Canvas
```

# Librerías

- **pandas** — los datos del caso y la traza del flujo se leen en tabla, no de memoria. Es la única librería que este laboratorio necesita: si carga y los datos se leen, verificaste la base de este laboratorio; eso no acredita todas las dependencias de los siguientes.

In [ ]:
# Este laboratorio utiliza pandas, disponible en el entorno habitual de Colab.
# Si la celda siguiente indica que falta, consulta la guía de Requisitos técnicos.
# No se instala software ni se llama a servicios externos desde este notebook.

In [1]:
from __future__ import annotations

import os                          # verificar rutas locales
import math                        # comprobar valores finitos
from dataclasses import dataclass  # estructura de la traza

import pandas as pd                # los datos del caso en tabla

print(f"pandas {pd.__version__} listo")

pandas 2.2.3 listo


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Preparación del entorno

La función siguiente carga los datos del caso y comprueba las columnas, los valores necesarios para el cálculo y las referencias entre tablas, con mensajes que te dicen exactamente qué hacer si algo falta: los errores útiles también son una convención de calidad que verás en todo el curso.

In [9]:
RUTA_DATOS = "/content/drive/MyDrive/Agentes/datos_muestra"
COLUMNAS_REQUERIDAS = {
    "ventas.csv": {"fecha", "tienda_id", "producto_id"},
    "inventarios.csv": {"fecha", "tienda_id", "producto_id", "unidades_disponibles"},
    "tiendas.csv": {"tienda_id", "zona"},
    "productos.csv": {"producto_id", "rotacion_esperada", "precio_lista"},
}


def cargar_datos(ruta: str = RUTA_DATOS) -> dict[str, pd.DataFrame]:
    """Lee archivos locales y comprueba lo necesario para estos cálculos.

    No descarga, modifica ni sustituye datos. Acepta el nombre de carpeta
    'datos_muestra' que puede producir la descompresión del ZIP de Canvas.
    """
    if ruta == "muestra" and not os.path.isdir(ruta) and os.path.isdir("datos_muestra"):
        ruta = "datos_muestra"
    if not os.path.isdir(ruta):
        raise FileNotFoundError(
            f"No encuentro la carpeta {ruta!r}. Descarga 'Datos de muestra' de Canvas "
            "y descomprime sus CSV en una carpeta 'muestra' o 'datos_muestra'. "
            "En Jupyter, colócala junto al notebook; en Colab, súbela al panel de archivos.")
    tablas: dict[str, pd.DataFrame] = {}
    for archivo, columnas in COLUMNAS_REQUERIDAS.items():
        ubicacion = os.path.join(ruta, archivo)
        if not os.path.isfile(ubicacion):
            raise FileNotFoundError(f"Falta {ubicacion}. Revisa la carpeta descomprimida de Canvas.")
        marco = pd.read_csv(ubicacion)
        faltantes = columnas - set(marco.columns)
        if faltantes:
            raise ValueError(f"{archivo}: faltan columnas {sorted(faltantes)}. Revisa la descarga.")
        if marco.empty or marco[list(columnas)].isna().any().any():
            raise ValueError(f"{archivo}: tabla vacía o valores requeridos ausentes. Revisa la descarga.")
        for columna in columnas & {"tienda_id", "producto_id", "zona"}:
            if marco[columna].astype(str).str.strip().eq("").any():
                raise ValueError(f"{archivo}: hay valores vacíos en {columna}.")
        if "fecha" in columnas:
            marco["fecha"] = pd.to_datetime(marco["fecha"], errors="coerce")
            if marco["fecha"].isna().any():
                raise ValueError(f"{archivo}: hay fechas inválidas. Revisa los datos de origen.")
            marco["fecha"] = marco["fecha"].dt.normalize()
        for columna in columnas & {"unidades_disponibles", "rotacion_esperada", "precio_lista"}:
            marco[columna] = pd.to_numeric(marco[columna], errors="coerce")
            if not marco[columna].map(math.isfinite).all() or marco[columna].lt(0).any():
                raise ValueError(f"{archivo}: {columna} debe contener números finitos no negativos.")
        tablas[archivo.split(".")[0]] = marco
    for tabla, clave in (("tiendas", "tienda_id"), ("productos", "producto_id")):
        if tablas[tabla][clave].duplicated().any():
            raise ValueError(f"{tabla}.csv: {clave} debe ser único; revisa los duplicados.")
    for tabla in ("ventas", "inventarios"):
        for catalogo, clave in (("tiendas", "tienda_id"), ("productos", "producto_id")):
            if not tablas[tabla][clave].isin(tablas[catalogo][clave]).all():
                raise ValueError(f"{tabla}.csv: hay {clave} que no existen en {catalogo}.csv.")
    return tablas


datos = cargar_datos()
ventas = datos["ventas"]
inventarios = datos["inventarios"]
tiendas = datos["tiendas"]
productos = datos["productos"]
print(f"Entorno listo para este laboratorio. {len(ventas):,} filas de ventas, "
      f"{len(tiendas)} tiendas, {len(productos)} productos.")

Entorno listo para este laboratorio. 12,632 filas de ventas, 40 tiendas, 20 productos.


**Qué deberías ver:** el mensaje "Entorno listo" con los conteos de tu descarga. Si en su lugar ves un error, léelo completo: te dice qué archivo o carpeta falta y dónde conseguirlo. Ese mensaje es tu primera entrega al hilo de dudas si no logras resolverlo.

## Sección A — El flujo que vas a observar

### El concepto

El Aprende distingue la ejecución de una ruta fija de la elección de pasos hacia un **objetivo**. Aquí los cinco pasos están programados: leer, detectar, calcular, contrastar y proponer. Llamar RAZONAR a un paso no demuestra deliberación de un modelo. La simulación **prepara el caso y lo deja en manos del gerente**.

```
  DEL ENCARGO A LA PROPUESTA (ruta programada, no elegida por un agente)

  "detecta el quiebre mas costoso     +-----------+   +----------+   +---------+
   y prepara el caso para el     -->  | PERCIBIR  |-->| DETECTAR |-->| RAZONAR |
   gerente de zona"                   +-----------+   +----------+   +---------+
                                                                          |
                        GERENTE  <--  PROPONER  <--  INVESTIGAR  <--------+
                       (decide el     (no ejecuta:    (¿aislado o
                        humano)        propone)        de zona?)
```

### La técnica: la traza tipada

Cada paso del flujo queda registrado con la misma forma — número, acción, detalle y resultado. Registrarlo así no es burocracia: es lo que permite auditar después qué hizo y por qué, la disciplina central del Skill Lab 2.

In [10]:
@dataclass(frozen=True)
class Paso:
    """Un paso de la traza del flujo: qué hizo, cómo y con qué resultado."""
    numero: int
    accion: str
    detalle: str
    resultado: str


class TrazaAgente:
    """Registro ordenado de los pasos programados del flujo, legible en vivo y en tabla."""

    def __init__(self) -> None:
        self.pasos: list[Paso] = []

    def registrar(self, accion: str, detalle: str, resultado: str) -> None:
        """Agrega un paso a la traza y lo muestra en el momento."""
        paso = Paso(len(self.pasos) + 1, accion, detalle, resultado)
        self.pasos.append(paso)
        print(f"[{paso.numero}] {paso.accion}\n    {paso.detalle}\n    -> {paso.resultado}\n")

    def como_tabla(self) -> pd.DataFrame:
        """Presenta la traza completa como tabla legible."""
        return pd.DataFrame(
            {"accion": [p.accion for p in self.pasos],
             "detalle": [p.detalle for p in self.pasos],
             "resultado": [p.resultado for p in self.pasos]},
            index=pd.RangeIndex(1, len(self.pasos) + 1, name="paso"))

In [11]:
# Flujo programado: cinco pasos fijos; no interpreta objetivos ni llama a una API.
traza = TrazaAgente()

# Ventana de siete fechas, anclada en la última fecha de ventas disponible.
fin = ventas.fecha.max()
semana = fin - pd.Timedelta(days=6)
inv_semana = inventarios[inventarios.fecha.between(semana, fin)]
traza.registrar(
    "PERCIBIR", f"Leo la ventana {semana.date()} a {fin.date()}, inclusive",
    f"{ventas.fecha.between(semana, fin).sum():,} filas de venta; "
    f"{len(inv_semana):,} de inventario")

# Una fecha se cuenta una sola vez aunque tenga varias observaciones en cero.
inv_cero = inv_semana[inv_semana.unidades_disponibles == 0]
dias_cero = inv_cero.groupby(["tienda_id", "producto_id"]).fecha.nunique()
traza.registrar(
    "DETECTAR", "Busco observaciones en cero; no prueban un quiebre continuo",
    f"{len(dias_cero)} combinaciones tienda-producto con al menos una fecha en cero; "
    f"{dias_cero.ge(2).sum()} con dos o más fechas distintas")

candidatos = (
    inv_cero
    .merge(productos[["producto_id", "rotacion_esperada", "precio_lista"]],
           on="producto_id", validate="many_to_one")
    .assign(costo_diario=lambda d: d.rotacion_esperada / 7 * d.precio_lista)
    .groupby(["tienda_id", "producto_id"])
    .agg(dias=("fecha", "nunique"), costo_diario=("costo_diario", "first"))
    .assign(costo_estimado=lambda d: d.dias * d.costo_diario)
    .sort_values("costo_estimado", ascending=False, kind="stable"))
caso = candidatos.head(1)

if caso.empty:
    traza.registrar("RAZONAR", "No hay candidatos para calcular un máximo",
                   "No se observaron inventarios en cero en la ventana disponible")
    traza.registrar("INVESTIGAR", "No hay un caso para comparar por producto y zona",
                   "La ausencia de candidatos no descarta problemas fuera de estos datos")
    traza.registrar("PROPONER", "No propongo una reposición",
                   "Revisar la cobertura y actualidad de los datos antes de concluir")
else:
    tienda_caso, producto_caso = caso.index[0]
    traza.registrar(
        "RAZONAR", "Aproximo costo: fechas en cero × rotación semanal / 7 × precio",
        f"Máximo estimado: {producto_caso} en {tienda_caso}, {int(caso.dias.iloc[0])} fechas, "
        f"${caso.costo_estimado.iloc[0]:,.0f}. No es venta perdida comprobada; "
        "usa rotación de referencia para tienda tamaño B, sin ajuste por tamaño")

    zonas = tiendas.set_index("tienda_id")["zona"]
    zona_caso = zonas.loc[tienda_caso]
    tiendas_zona = zonas[zonas == zona_caso].index
    mismas = candidatos[
        (candidatos.index.get_level_values("producto_id") == producto_caso)
        & candidatos.index.get_level_values("tienda_id").isin(tiendas_zona)]
    traza.registrar(
        "INVESTIGAR", "Cuento tiendas del mismo producto y de la misma zona",
        f"{len(mismas)} tiendas de la zona {zona_caso}, incluida la del caso, "
        "tienen observaciones en cero en la ventana. No demuestra simultaneidad "
        "ni una falla del centro de distribución")
    traza.registrar(
        "PROPONER", "Muestro una propuesta para revisión humana; no envío ni ejecuto órdenes",
        f"Revisar existencias y abastecimiento de {producto_caso} en {tienda_caso}; "
        "verificar la causa y la necesidad antes de autorizar una reposición")

print(f"Traza completa: {len(traza.pasos)} pasos programados. "
      "Solo se leyeron archivos y se mostraron resultados en este notebook.")
display(traza.como_tabla())

[1] PERCIBIR
    Leo la ventana 2026-06-22 a 2026-06-28, inclusive
    -> 3,057 filas de venta; 2,440 de inventario

[2] DETECTAR
    Busco observaciones en cero; no prueban un quiebre continuo
    -> 193 combinaciones tienda-producto con al menos una fecha en cero; 111 con dos o más fechas distintas

[3] RAZONAR
    Aproximo costo: fechas en cero × rotación semanal / 7 × precio
    -> Máximo estimado: P007 en T0004, 6 fechas, $1,723. No es venta perdida comprobada; usa rotación de referencia para tienda tamaño B, sin ajuste por tamaño

[4] INVESTIGAR
    Cuento tiendas del mismo producto y de la misma zona
    -> 2 tiendas de la zona Centro, incluida la del caso, tienen observaciones en cero en la ventana. No demuestra simultaneidad ni una falla del centro de distribución

[5] PROPONER
    Muestro una propuesta para revisión humana; no envío ni ejecuto órdenes
    -> Revisar existencias y abastecimiento de P007 en T0004; verificar la causa y la necesidad antes de autorizar una reposic

,accion,detalle,resultado
paso,,,
1,PERCIBIR,"Leo la ventana 2026-06-22 a 2026-06-28, inclusive","3,057 filas de venta; 2,440 de inventario"
2,DETECTAR,Busco observaciones en cero; no prueban un qui...,193 combinaciones tienda-producto con al menos...
3,RAZONAR,Aproximo costo: fechas en cero × rotación sema...,"Máximo estimado: P007 en T0004, 6 fechas, $1,7..."
4,INVESTIGAR,Cuento tiendas del mismo producto y de la mism...,"2 tiendas de la zona Centro, incluida la del c..."
5,PROPONER,Muestro una propuesta para revisión humana; no...,Revisar existencias y abastecimiento de P007 e...


**Qué deberías ver:** cinco pasos numerados y, al final, la traza como tabla. Fíjate en dos cosas. El **orden** está fijado en el código. Y el **final** no envía una orden ni modifica inventarios: muestra texto. Además, revisa el alcance del cálculo: INVESTIGAR cuenta tiendas del mismo producto y de la misma zona. Contar coincidencias no demuestra una falla del centro de distribución ni justifica por sí solo una reposición regional. Los días son fechas distintas con observaciones en cero, no necesariamente días consecutivos ni días completos sin inventario. El costo usa una rotación semanal de referencia para tienda tamaño B: es una aproximación, no venta perdida comprobada. Distingue el dato observado, la hipótesis causal y la decisión que requiere evidencia adicional.

### El método: las dos preguntas del curso

Cada sistema que veas — en este curso y en tu organización — se examina con dos preguntas: **¿qué decide?** y **¿qué puede tocar?** Este flujo selecciona un caso con una fórmula y una ruta definidas por quien lo programó. Lee archivos y muestra una propuesta en el notebook; no la envía a una bandeja real ni cuenta con un mecanismo implementado de aprobación. Esa distancia entre lo que decide y lo que toca es la primera decisión de diseño de un sistema agéntico, y volverás a ella cada semana.

# Tu entrega (tres respuestas breves)

Responde estas tres preguntas en la celda siguiente, entre las comillas. El validador comprueba que haya texto en las tres respuestas; no evalúa tus argumentos ni determina si son correctos. Conserva tu notebook de trabajo y prepara un PDF con las tres respuestas y evidencia legible de ejecución: mensaje de carga de datos, traza y resultado de las comprobaciones técnicas. Puedes integrar capturas de esas salidas y tus respuestas en un documento y exportarlo como PDF. Antes de subirlo, abre el archivo y comprueba que todo se lea, sin recortes. Nómbralo Laboratorio0_#matrícula.pdf. No entregues únicamente una liga: se revisará el contenido del PDF enviado. Se marca completo con la entrega; no se califica.

1. ¿Qué pasos están programados y qué evidencia necesitarías para afirmar que el sistema elige su ruta?
2. ¿Qué información puede consultar, qué acción puede realizar y cuál requeriría autorización humana?
3. ¿Qué conclusión de la traza aceptarías y cuál pedirías verificar antes de autorizar una reposición? Justifica tu respuesta con lo que el código sí comprueba.

In [12]:
# DECISIÓN | Tus tres respuestas, entre las comillas
respuesta_1 = """
Los pasos programados son PERCIBIR, DETECTAR, RAZONAR, INVESTIGAR y PROPONER. El código fija su orden, sus cálculos y la bifurcación cuando no hay candidatos. Por eso, la traza demuestra la ejecución de una ruta fija, no que el sistema haya elegido su propia ruta. Para afirmar que la elige, necesitaría ver las alternativas consideradas, los criterios utilizados y por qué seleccionó unos pasos y descartó otros.
"""

respuesta_2 = """
El sistema puede consultar los archivos locales de ventas, inventarios, tiendas y productos. Puede revisar una ventana de siete fechas, detectar observaciones de inventario cero, calcular un costo estimado, comparar tiendas de la misma zona y mostrar una propuesta. No modifica inventarios ni genera o envía órdenes. La autorización y ejecución de una reposición deben quedar a cargo de una persona.
"""

respuesta_3 = """
Aceptaría que la combinación tienda-producto seleccionada tuvo observaciones de inventario cero y obtuvo el mayor costo estimado mediante la fórmula programada. Antes de autorizar una reposición verificaría las existencias actuales, la duración real del quiebre, la demanda no atendida, el abastecimiento en tránsito y la causa. El código solo cuenta fechas distintas con inventario cero; no demuestra un quiebre continuo, ventas perdidas reales, simultaneidad en la zona ni una falla del centro de distribución.
"""


def validar_respuestas(respuestas: list[str]) -> list[int]:
    """Comprueba presencia de texto; no evalúa la calidad de los argumentos."""
    return [n for n, r in enumerate(respuestas, start=1)
            if not isinstance(r, str) or not r.strip()]


pendientes = validar_respuestas([respuesta_1, respuesta_2, respuesta_3])
for n in (1, 2, 3):
    estado = "PENDIENTE" if n in pendientes else "TEXTO REGISTRADO (sin evaluar)"
    print(f"Respuesta {n}: {estado}")
if pendientes:
    print("Completa las respuestas pendientes antes de preparar y subir tu PDF a Canvas.")
else:
    print("Hay texto en las tres respuestas. Revisa tus argumentos y sube el PDF con tus respuestas y evidencia de ejecución.")

Respuesta 1: TEXTO REGISTRADO (sin evaluar)
Respuesta 2: TEXTO REGISTRADO (sin evaluar)
Respuesta 3: TEXTO REGISTRADO (sin evaluar)
Hay texto en las tres respuestas. Revisa tus argumentos y sube el PDF con tus respuestas y evidencia de ejecución.


## El notebook se verifica a sí mismo

La celda siguiente corre **pruebas automáticas** de algunos aspectos del instrumento: columnas, secuencia, máximo calculado y campos de respuesta. También admiten que no haya candidatos: en ese caso, la traza lo indica y no propone una reposición. No validan tus argumentos ni la causa del quiebre, ni demuestran que no exista un problema fuera de los datos observados.

In [13]:
def pruebas_del_notebook() -> None:
    """Cuatro grupos de comprobaciones técnicas, no una evaluación académica."""
    # 1. Se cargaron las tablas y las columnas requeridas.
    for nombre, columnas in COLUMNAS_REQUERIDAS.items():
        tabla = datos[nombre.split(".")[0]]
        assert columnas <= set(tabla.columns), f"{nombre}: faltan columnas"
        assert not tabla.empty, f"{nombre}: llegó vacío"

    # 2. La secuencia es fija y explícita, incluso si no hay candidatos.
    assert [p.accion for p in traza.pasos] == [
        "PERCIBIR", "DETECTAR", "RAZONAR", "INVESTIGAR", "PROPONER"]

    # 3. Máximo, días distintos y comparación dentro de la zona.
    assert caso.empty == candidatos.empty
    if not caso.empty:
        assert caso.costo_estimado.iloc[0] == candidatos.costo_estimado.max()
        assert candidatos.dias.between(1, 7).all()
        assert int(caso.dias.iloc[0]) == int(dias_cero.loc[caso.index[0]])
        assert mismas.index.get_level_values("tienda_id").isin(tiendas_zona).all()

    # 4. Detectar texto no equivale a comprobar una respuesta correcta.
    assert validar_respuestas(["", " ", "x"]) == [1, 2]
    assert validar_respuestas([None, 12, "texto"]) == [1, 2]
    assert validar_respuestas(["a", "b", "c"]) == []
    print("4/4 grupos de comprobaciones técnicas en verde. "
          "No validan tus argumentos, la causa del quiebre ni una decisión de reposición.")


pruebas_del_notebook()

4/4 grupos de comprobaciones técnicas en verde. No validan tus argumentos, la causa del quiebre ni una decisión de reposición.


# Conclusiones

Cierra tu primera sesión con el caso (una línea cada una):

* **Sobre el entorno:** ¿corrió todo a la primera? Si no, ¿qué mensaje de error te llevó a la solución?
* **Sobre la traza:** ¿qué paso te sorprendió más y por qué?

---
**Qué sigue:** en el Skill Lab 1 dejarás de observar y empezarás a decidir: qué modelo conviene a cada tarea del caso, con números enfrente. Las dos preguntas de hoy — qué decide, qué puede tocar — te acompañan todo el curso, hasta el expediente de liberación del capstone.